**Status: Future-race podium probability.** Predicts each driver's chance of a top-3 finish from the grid and recent form, for a race that has not happened yet, including drivers and teams that were never in the training data.

Reads `f1_race_results` (from `phase_1b_results.ipynb`). Features use only races before the one being predicted: recency-weighted team pace, team qualifying gap, team reliability, the driver's edge over teammates, circuit mobility, and the grid. Car pace resets after the 2022 and 2026 regulation changes (earlier seasons get a small weight). Output probabilities are normalised so a race sums to 3 (three podium places).

The model ships only if it beats the grid-slot baseline on walk-forward folds with a race-clustered bootstrap CI that excludes zero; otherwise the App shows the baseline.

In [ ]:
import json

import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, brier_score_loss, log_loss

rr = spark.table("workspace.default.f1_race_results").toPandas()
rr["EventDate"] = pd.to_datetime(rr["EventDate"])
rr = rr.sort_values(["EventDate", "Round", "Driver"]).reset_index(drop=True)
rr["RaceIdx"] = rr.groupby(["Year", "Race"], sort=False).ngroup()

finished = rr["Status"].str.contains(r"Finished|^\+\d+ Lap|Lapped", regex=True)
rr["Dnf"] = (~finished).astype(float)
rr["FinScore"] = np.where(finished & rr["FinalPosition"].notna(), 21 - rr["FinalPosition"], 0.0)
rr["Podium"] = (rr["FinalPosition"] <= 3).astype(int)
rr["Move"] = np.where(finished, (rr["FinalPosition"] - rr["GridPosition"]).abs(), np.nan)

ASOF = rr["EventDate"].max()
LATEST = rr.iloc[-1]
print("Races:", rr["RaceIdx"].nunique(), "| rows:", len(rr), "| latest:", LATEST["Year"], LATEST["Race"], ASOF.date())

### Step 1: Leak-free form features (recency weighted, reset-aware)

In [ ]:
RESET_YEARS = {2022, 2026}
WINDOW = 4
DECAY_OLD = 0.2
DECAY_RESET = 0.1
C_REG = 0.1

def trailing(values, years):
    # weighted mean of the previous WINDOW races: same season weight 1, older seasons DECAY_OLD (DECAY_RESET across a regulation reset)
    out = np.full(len(values), np.nan)
    support = np.zeros(len(values))
    for i in range(len(values)):
        lo = max(0, i - WINDOW)
        v, y = values[lo:i], years[lo:i]
        if len(v) == 0:
            continue
        w = np.where(y == years[i], 1.0, DECAY_RESET if years[i] in RESET_YEARS else DECAY_OLD)
        ok = ~np.isnan(v)
        if ok.any():
            out[i] = (w[ok] * v[ok]).sum() / w[ok].sum()
            support[i] = w[ok].sum()
    return out, support

def add_features(df):
    d = df.copy()
    d["Edge"] = d["FinScore"] - d.groupby(["RaceIdx", "TeamKey"])["FinScore"].transform("mean")
    team = d.groupby(["TeamKey", "RaceIdx", "Year"], as_index=False).agg(
        TeamFin=("FinScore", "mean"), TeamQ=("QualiGapPct", "mean"), TeamDnfRate=("Dnf", "mean"))
    parts = []
    for _, g in team.sort_values("RaceIdx").groupby("TeamKey"):
        g = g.copy()
        yrs = g["Year"].to_numpy()
        g["TeamPace"], g["TeamSupport"] = trailing(g["TeamFin"].to_numpy(float), yrs)
        g["TeamQuali"], _ = trailing(g["TeamQ"].to_numpy(float), yrs)
        g["TeamDnf"], _ = trailing(g["TeamDnfRate"].to_numpy(float), yrs)
        parts.append(g[["TeamKey", "RaceIdx", "TeamPace", "TeamSupport", "TeamQuali", "TeamDnf"]])
    d = d.merge(pd.concat(parts), on=["TeamKey", "RaceIdx"], how="left")
    dparts = []
    for _, g in d.sort_values("RaceIdx").groupby("Driver"):
        g = g.copy()
        g["DriverEdge"], _ = trailing(g["Edge"].to_numpy(float), g["Year"].to_numpy())
        g["DriverRaces"] = np.arange(len(g))
        dparts.append(g[["Driver", "RaceIdx", "DriverEdge", "DriverRaces"]])
    d = d.merge(pd.concat(dparts), on=["Driver", "RaceIdx"], how="left")
    race_move = d.groupby(["Location", "RaceIdx"], as_index=False)["Move"].mean().sort_values("RaceIdx")
    race_move["CircuitMobility"] = race_move.groupby("Location")["Move"].transform(lambda s: s.shift(1).expanding().mean())
    return d.merge(race_move[["Location", "RaceIdx", "CircuitMobility"]], on=["Location", "RaceIdx"], how="left")

# snapshot rows: one synthetic "next race" row per recent driver, so the same code yields as-of-today form
recent = rr[rr["RaceIdx"] >= rr["RaceIdx"].max() - 2].sort_values("RaceIdx").drop_duplicates("Driver", keep="last")
nxt = recent[["Driver", "FullName", "TeamName", "TeamKey", "DriverNumber"]].copy()
nxt["Year"], nxt["RaceIdx"], nxt["Location"], nxt["CircuitName"], nxt["Race"] = int(LATEST["Year"]), rr["RaceIdx"].max() + 1, "NEXT", "NEXT", "NEXT"
nxt["EventDate"] = ASOF + pd.Timedelta(days=14)
for c in ("FinScore", "Dnf", "QualiGapPct", "Move", "GridPosition", "FinalPosition", "Podium"):
    nxt[c] = np.nan

full = add_features(pd.concat([rr, nxt], ignore_index=True))
snap = full[full["Race"] == "NEXT"].copy()
hist = full[full["Race"] != "NEXT"].copy()
print("Snapshot drivers:", len(snap), "| teams:", snap["TeamKey"].nunique())

### Step 2: Fills, model fit and per-race normalisation to 3 podium places

In [ ]:
FEATS = ["LogGrid", "SlotLogit", "QualiGapPct", "TeamPace", "TeamQuali", "TeamDnf", "DriverEdge", "CircuitMobility"]
TRAIN_FROM_YEAR = 2019

def make_fills(tr):
    return {
        "gap_by_slot": tr.groupby("GridPosition")["QualiGapPct"].median().to_dict(),
        "gap_global": float(tr["QualiGapPct"].median()),
        "slot_rate": tr.groupby("GridPosition")["Podium"].mean().to_dict(),
        "TeamPace": float(tr["TeamPace"].quantile(0.10)),
        "TeamQuali": float(tr["TeamQuali"].quantile(0.90)),
        "TeamDnf": float(tr["TeamDnf"].median()),
        "DriverEdge": float(tr.loc[tr["DriverRaces"] <= 2, "DriverEdge"].median()),
        "CircuitMobility": float(tr["CircuitMobility"].median()),
    }

def apply_fills(df, f):
    d = df.copy()
    d["TeamNew"] = d["TeamPace"].isna() | (d["TeamSupport"].fillna(0) < 2.0)
    d["DriverNew"] = d["DriverRaces"] < 3
    slot_gap = d["GridPosition"].map(lambda s: f["gap_by_slot"].get(s, f["gap_global"]) if pd.notna(s) else f["gap_global"])
    d["QualiGapPct"] = d["QualiGapPct"].fillna(slot_gap)
    for c in ("TeamPace", "TeamQuali", "TeamDnf", "DriverEdge", "CircuitMobility"):
        d[c] = d[c].fillna(f[c])
    d["LogGrid"] = np.log(d["GridPosition"].clip(lower=1))
    last = f["slot_rate"][max(f["slot_rate"])]
    rate = d["GridPosition"].map(lambda s: f["slot_rate"].get(s, last)).fillna(last).clip(0.01, 0.99)
    d["SlotLogit"] = np.log(rate / (1 - rate))
    return d

def fit_lr(tr, feats=FEATS, c=None):
    c = C_REG if c is None else c
    mu, sd = tr[feats].mean(), tr[feats].std().replace(0, 1.0)
    lr = LogisticRegression(C=c, max_iter=3000).fit((tr[feats] - mu) / sd, tr["Podium"])
    return lr, mu, sd

def logit_scores(lr, mu, sd, d, feats=FEATS):
    return ((d[feats] - mu) / sd).to_numpy() @ lr.coef_[0] + lr.intercept_[0]

def norm3(z):
    # shift all logits by one constant so the field's podium probabilities sum to 3
    lo, hi = -15.0, 15.0
    for _ in range(60):
        mid = (lo + hi) / 2
        if (1 / (1 + np.exp(-(z + mid)))).sum() > 3:
            hi = mid
        else:
            lo = mid
    return 1 / (1 + np.exp(-(z + (lo + hi) / 2)))

### Step 3: Walk-forward backtest vs grid-slot baselines (2022 to latest)

In [ ]:
usable = hist[hist["Year"] >= TRAIN_FROM_YEAR].copy()
usable = usable[usable["FinalPosition"].notna()]
test_years = sorted(y for y in usable["Year"].unique() if y >= 2022)

preds = []
for ty in test_years:
    tr_raw, te_raw = usable[usable["Year"] < ty], usable[usable["Year"] == ty]
    fills = make_fills(tr_raw)
    tr, te = apply_fills(tr_raw, fills), apply_fills(te_raw, fills)
    slot_rate = tr.groupby("GridPosition")["Podium"].mean()
    te = te.assign(
        slot=te["GridPosition"].map(lambda s: slot_rate.get(min(s, slot_rate.index.max()), slot_rate.iloc[-1])).to_numpy(),
    )
    lr_g, mu_g, sd_g = fit_lr(tr, ["LogGrid"])
    lr_m, mu_m, sd_m = fit_lr(tr)
    te["logit_grid"] = 1 / (1 + np.exp(-logit_scores(lr_g, mu_g, sd_g, te, ["LogGrid"])))
    z = logit_scores(lr_m, mu_m, sd_m, te)
    te["model_raw"] = 1 / (1 + np.exp(-z))
    te["model"] = np.nan
    for _, idx in te.groupby("RaceIdx").groups.items():
        te.loc[idx, "model"] = norm3(z[te.index.get_indexer(idx)])
    preds.append(te)
bt = pd.concat(preds)

def metrics(df, col):
    return dict(auc=roc_auc_score(df["Podium"], df[col]), brier=brier_score_loss(df["Podium"], df[col]), logloss=log_loss(df["Podium"], df[col].clip(1e-4, 1 - 1e-4)))

COLS = ["slot", "logit_grid", "model_raw", "model"]
rows = []
for ty in test_years:
    for col in COLS:
        rows.append(dict(Year=ty, Model=col, Races=bt[bt["Year"] == ty]["RaceIdx"].nunique(), **metrics(bt[bt["Year"] == ty], col)))
rows += [dict(Year=0, Model=col, Races=bt["RaceIdx"].nunique(), **metrics(bt, col)) for col in COLS]
backtest = pd.DataFrame(rows)
print(backtest.round(4).to_string(index=False))
print("Mean podium-probability sum per race (model):", round(bt.groupby("RaceIdx")["model"].sum().mean(), 3))

### Step 4: Ship rule (race-clustered bootstrap)

In [ ]:
rng = np.random.default_rng(0)
race_ids = bt["RaceIdx"].to_numpy()
races = np.unique(race_ids)
by_race = {r: np.where(race_ids == r)[0] for r in races}
y = bt["Podium"].to_numpy()

def boot_delta(base, model, n=1000):
    d_b, d_a = [], []
    for _ in range(n):
        pk = np.concatenate([by_race[r] for r in rng.choice(races, len(races))])
        d_b.append(brier_score_loss(y[pk], bt[base].to_numpy()[pk]) - brier_score_loss(y[pk], bt[model].to_numpy()[pk]))
        d_a.append(roc_auc_score(y[pk], bt[model].to_numpy()[pk]) - roc_auc_score(y[pk], bt[base].to_numpy()[pk]))
    return np.mean(d_b), *np.percentile(d_b, [2.5, 97.5]), np.mean(d_a), *np.percentile(d_a, [2.5, 97.5])

for base in ("slot", "logit_grid"):
    b, blo, bhi, a, alo, ahi = boot_delta(base, "model")
    print(f"model vs {base}: dBrier {b:+.5f} [{blo:+.5f}, {bhi:+.5f}] | dAUC {a:+.4f} [{alo:+.4f}, {ahi:+.4f}]")
    if base == "slot":
        SHIP_PODIUM = blo > 0
print("Ship rule passed (CI of Brier gain over grid-slot baseline excludes zero):", SHIP_PODIUM)

bins = pd.cut(bt["model"], [0, .02, .05, .1, .2, .4, .6, .8, 1.0])
calib = bt.groupby(bins, observed=True).agg(pred=("model", "mean"), observed=("Podium", "mean"), n=("Podium", "size"))
print(calib.round(3))

### Step 5: Final model, bootstrap ensemble for ranges, snapshot tables

In [ ]:
fills_f = make_fills(usable)
tr_f = apply_fills(usable, fills_f)
lr_f, mu_f, sd_f = fit_lr(tr_f)

ens = []
r_all = tr_f["RaceIdx"].unique()
by_r = {r: np.where(tr_f["RaceIdx"].to_numpy() == r)[0] for r in r_all}
for _ in range(40):
    pk = np.concatenate([by_r[r] for r in rng.choice(r_all, len(r_all))])
    m, _, _ = fit_lr(tr_f.iloc[pk])
    ens.append(dict(coef=m.coef_[0].tolist(), intercept=float(m.intercept_[0])))

slot_table = tr_f.groupby("GridPosition")["Podium"].mean()
model_json = {
    "features": FEATS, "mu": mu_f.to_dict(), "sd": sd_f.to_dict(),
    "coef": lr_f.coef_[0].tolist(), "intercept": float(lr_f.intercept_[0]), "ensemble": ens,
    "fills": {k: v for k, v in fills_f.items() if k not in ("gap_by_slot", "slot_rate")},
    "gap_by_slot": {str(int(k)): float(v) for k, v in fills_f["gap_by_slot"].items()},
    "slot_table": {str(int(k)): float(v) for k, v in slot_table.items()},
    "ship_model": bool(SHIP_PODIUM), "asof": str(ASOF.date()), "latest_year": int(LATEST["Year"]), "latest_race": str(LATEST["Race"]),
    "pooled": backtest[backtest["Year"] == 0].set_index("Model")[["auc", "brier", "logloss"]].round(5).to_dict("index"),
}
spark.createDataFrame(pd.DataFrame({"Json": [json.dumps(model_json)]})).write.mode("overwrite").saveAsTable("workspace.default.f1_podium_model")

snap_f = apply_fills(snap.assign(GridPosition=np.nan, QualiGapPct=np.nan), fills_f)
season_races = hist[hist["Year"] == LATEST["Year"]].groupby("TeamKey")["RaceIdx"].nunique()
drivers_tbl = snap_f[["Driver", "FullName", "TeamName", "TeamKey", "DriverNumber", "DriverEdge", "DriverRaces", "DriverNew"]].copy()
drivers_tbl["Current"] = drivers_tbl["Driver"].isin(rr.loc[rr["RaceIdx"] == rr["RaceIdx"].max(), "Driver"])
teams_tbl = (
    snap_f.drop_duplicates("TeamKey")[["TeamKey", "TeamPace", "TeamQuali", "TeamDnf", "TeamSupport", "TeamNew"]]
    .assign(SeasonRaces=lambda x: x["TeamKey"].map(season_races).fillna(0).astype(int))
)
circuits_tbl = hist.drop_duplicates("Location", keep="last")[["Location", "CircuitName"]].merge(
    hist.groupby("Location", as_index=False)["Move"].mean().rename(columns={"Move": "CircuitMobility"}), on="Location")
spark.createDataFrame(drivers_tbl).write.mode("overwrite").saveAsTable("workspace.default.f1_podium_driver_form")
spark.createDataFrame(teams_tbl).write.mode("overwrite").saveAsTable("workspace.default.f1_podium_team_form")
spark.createDataFrame(circuits_tbl).write.mode("overwrite").saveAsTable("workspace.default.f1_podium_circuits")
spark.createDataFrame(backtest).write.mode("overwrite").saveAsTable("workspace.default.f1_podium_backtest")
last_grids = hist[hist["Year"] == LATEST["Year"]][["Year", "Race", "Round", "Driver", "TeamKey", "GridPosition"]]
spark.createDataFrame(last_grids).write.mode("overwrite").saveAsTable("workspace.default.f1_podium_recent_grids")
print(teams_tbl.sort_values("TeamPace", ascending=False).round(2).to_string(index=False))
print("Coefs:", dict(zip(FEATS, np.round(lr_f.coef_[0], 3))))

In [ ]:
def predict_field(entries, circuit_mobility=None):
    # entries: [{"driver","team","grid","gap"(optional)}] -> list of dicts with p, lo, hi; same logic the App uses
    d = snap_f.set_index("Driver")
    t = snap_f.drop_duplicates("TeamKey").set_index("TeamKey")
    rows = []
    for e in entries:
        drv = d.loc[e["driver"]] if e["driver"] in d.index else None
        tm = t.loc[e["team"]] if e["team"] in t.index else None
        rows.append(dict(
            GridPosition=e["grid"], LogGrid=np.log(max(e["grid"], 1)),
            SlotLogit=float(np.log(min(max(slot_table.get(e["grid"], slot_table.iloc[-1]), 0.01), 0.99) / (1 - min(max(slot_table.get(e["grid"], slot_table.iloc[-1]), 0.01), 0.99)))),
            QualiGapPct=e.get("gap") if e.get("gap") is not None else fills_f["gap_by_slot"].get(e["grid"], fills_f["gap_global"]),
            TeamPace=tm["TeamPace"] if tm is not None and pd.notna(tm["TeamPace"]) else fills_f["TeamPace"],
            TeamQuali=tm["TeamQuali"] if tm is not None and pd.notna(tm["TeamQuali"]) else fills_f["TeamQuali"],
            TeamDnf=tm["TeamDnf"] if tm is not None and pd.notna(tm["TeamDnf"]) else fills_f["TeamDnf"],
            DriverEdge=drv["DriverEdge"] if drv is not None and pd.notna(drv["DriverEdge"]) else fills_f["DriverEdge"],
            CircuitMobility=circuit_mobility if circuit_mobility is not None else fills_f["CircuitMobility"],
        ))
    x = pd.DataFrame(rows)
    z = ((x[FEATS] - mu_f) / sd_f).to_numpy() @ lr_f.coef_[0] + lr_f.intercept_[0]
    p = norm3(z)
    zs = np.array([((x[FEATS] - mu_f) / sd_f).to_numpy() @ np.array(m["coef"]) + m["intercept"] for m in ens])
    ps = np.array([norm3(zz) for zz in zs])
    return [dict(driver=e["driver"], p=float(p[i]), lo=float(np.percentile(ps[:, i], 10)), hi=float(np.percentile(ps[:, i], 90))) for i, e in enumerate(entries)]

latest_grid = last_grids[last_grids["Race"] == LATEST["Race"]].sort_values("GridPosition")
demo = predict_field([dict(driver=r.Driver, team=r.TeamKey, grid=int(r.GridPosition)) for r in latest_grid.itertuples()])
for r in demo[:8]:
    print(f"{r['driver']}: {r['p']:.0%} ({r['lo']:.0%}-{r['hi']:.0%})")
print("Sum:", round(sum(r["p"] for r in demo), 3))

### Limits

- Form is only as fresh as `f1_race_results`; re-run `phase_1b_results.ipynb` and this notebook after each race weekend.
- No weather, penalties or start-lap risk are modelled; wet races and grid penalties after qualifying are not in the grid slot.
- A new team or driver gets a back-of-grid or rookie prior and a wider range; the App flags them.
- After a regulation reset the first few races of the season have thin support, so ranges are wide until the new cars have raced.

## Out-of-sample pre-race probabilities for the live models

Walk-forward by season from 2020: each season's podium probabilities come from a model fitted only on earlier seasons, on the real grid. `phase_3_live.ipynb` uses them as the pre-race prior, so its training and backtest never see a prior that knew the result.

In [ ]:
prior_rows = []
for ty in range(TRAIN_FROM_YEAR + 1, int(usable["Year"].max()) + 1):
    tr_raw, te_raw = usable[usable["Year"] < ty], usable[usable["Year"] == ty]
    fills = make_fills(tr_raw)
    tr, te = apply_fills(tr_raw, fills), apply_fills(te_raw, fills)
    lr_y, mu_y, sd_y = fit_lr(tr)
    z = logit_scores(lr_y, mu_y, sd_y, te)
    te = te.assign(Prior=np.nan)
    for _, idx in te.groupby("RaceIdx").groups.items():
        te.loc[idx, "Prior"] = norm3(z[te.index.get_indexer(idx)])
    prior_rows.append(te[["Year", "Race", "Round", "Driver", "Prior", "GridPosition", "FinalPosition"]])
prior_oof = pd.concat(prior_rows, ignore_index=True)
spark.createDataFrame(prior_oof).write.mode("overwrite").saveAsTable("workspace.default.f1_podium_prior_oof")
print("Prior rows:", len(prior_oof), "| seasons:", sorted(prior_oof["Year"].unique()))